In [10]:
import torch
import torch.nn as nn

In [11]:
class LoRALinear(nn.Module):
    def __init__(self, base_layer, r, alpha):
        super().__init__()
        self.base_layer = base_layer
        self.A = nn.Parameter(torch.randn(r, base_layer.in_features))
        self.B = nn.Parameter(torch.zeros(base_layer.out_features, r))
        self.scale = alpha / r
        for param in base_layer.parameters():
            param.requires_grad = False

    def forward(self, x):
        self.BA = (self.B @ self.A) 
        frozen = self.base_layer(x)
        LoRA = (x @ self.BA.T) * self.scale

        return frozen + LoRA

In [12]:
base_model = nn.Linear(128, 22)

print(sum(p.numel() for p in base_model.parameters()if p.requires_grad)) 

2838


In [13]:
model = LoRALinear(nn.Linear(128, 22), 2, 10)

dummy = torch.randn(1, 256, 128)

out = model(dummy)
out2 = model.base_layer(dummy)

print(torch.allclose(out, out2))

print(sum(p.numel() for p in model.parameters()if p.requires_grad)) 

True
300


In [16]:
def compute_attention(Q, K, d_model):
    scores = Q @ K.transpose(-2, -1) / d_model**0.5

    return scores

class SingleHeadAttention(nn.Module):
    def __init__(self, d_model, r=2, alpha=10):
        super().__init__()
        self.d_model = d_model
        self.Wq = LoRALinear(nn.Linear(d_model, d_model), r=r, alpha=alpha)
        self.Wk = nn.Linear(d_model, d_model)
        self.Wv = LoRALinear(nn.Linear(d_model, d_model), r=r, alpha=alpha)
        self.Wo = nn.Linear(d_model, d_model)
        
        for param in self.Wk.parameters():
            param.requires_grad = False
            
        for param in self.Wo.parameters():
            param.requires_grad = False


    def forward(self, x):
        scores = compute_attention(self.Wq(x), self.Wk(x), self.d_model)

        weights = torch.softmax(scores, dim=-1)

        outputs = weights @ self.Wv(x)

        return self.Wo(outputs)


In [19]:
dummy_in = torch.randn(2, 5, 16)

head_lora = SingleHeadAttention(16)
out_lora = head_lora(dummy_in)

Q = head_lora.Wq.base_layer(dummy_in)
K = head_lora.Wk(dummy_in)
V = head_lora.Wv.base_layer(dummy_in)

scores = compute_attention(Q, K, d_model=head_lora.d_model)

weights = torch.softmax(scores, dim=-1)

outputs = weights @ V

manual_out = head_lora.Wo(outputs)

print(torch.allclose(manual_out, out_lora))

True


In [ ]:
A_before = head_lora.Wq.A.clone()
B_before = head_lora.Wq.B.clone()

Wk_weight = head_lora.Wk.weight.clone()
Wk_bias = head_lora.Wk.bias.clone()

Wo_weight = head_lora.Wo.weight.clone()
Wo_bias = head_lora.Wo.bias.clone()

Wq_weight = head_lora.Wq.base_layer.weight.clone()
Wq_bias = head_lora.Wq.base_layer.bias.clone()

Wv_weight = head_lora.Wv.base_layer.weight.clone()
Wv_bias = head_lora.Wv.base_layer.bias.clone()

out = head_lora(dummy_in)
target = torch.randn_like(out)

criterion = torch.nn.MSELoss()
optimizer = torch.optim.Adam([head_lora.Wq.A, head_lora.Wq.B, head_lora.Wv.A, head_lora.Wv.B])

optimizer.zero_grad()

loss = criterion(target, out)

print(f"Number of parameters in optimizer: {len(optimizer.param_groups[0]['params'])}")

print("Optimizer Parameters Check")
print(optimizer.param_groups[0]['params'][0] is head_lora.Wq.A)
print(optimizer.param_groups[0]['params'][1] is head_lora.Wq.B)
print(optimizer.param_groups[0]['params'][2] is head_lora.Wv.A)
print(optimizer.param_groups[0]['params'][3] is head_lora.Wv.B)

loss.backward()
optimizer.step()

print("Frozen Parameters Check")
print("K")
print(torch.equal(Wk_weight, head_lora.Wk.weight))
print(torch.equal(Wk_bias, head_lora.Wk.bias))

print("O")
print(torch.equal(Wo_weight, head_lora.Wo.weight))
print(torch.equal(Wo_bias, head_lora.Wo.bias))

print("Q")
print(torch.equal(Wq_weight, head_lora.Wq.base_layer.weight))
print(torch.equal(Wq_bias, head_lora.Wq.base_layer.bias))

print("V")
print(torch.equal(Wv_weight, head_lora.Wv.base_layer.weight))
print(torch.equal(Wv_bias, head_lora.Wv.base_layer.bias))

print("Requires Grad Parameters Check")
print(torch.equal(A_before, head_lora.Wq.A))
print(torch.equal(B_before, head_lora.Wq.B))


Number of parameters in optimizer: 4
Optimizer Parameters Check
True
True
True
True
Frozen Parameters Check
K
True
True
O
True
True
Q
True
True
V
True
True
Requires Grad Parameters Check
False
False
